## Topic:  Retrievers in LangChain

### Agenda
- 1. Introduction of Retrievers

- 2. Types of Retrievers in LangChain

- 3. How to create a Retriever in LangChain?

- 4. Important Retriever Parameters

- 5. Retriever Use Cases

- 6. Advantages and Limitations of Retrievers

- 7. Summary of Retrievers

### 1. Introduction of Retrievers

- Definition:
    - A Retriever is a component that receives a query and returns the most relevant documents or document chunks from a knowledge source.

    - A retriever is a component in LangChain that fetches relevant documents from a data source in response to a user’s query.

    - All retrievers in LangChain are runnables


- In simple words:
    - Retriever = A component that finds relevant information for a user's question.



In [ ]:
"""  Retrievers Workflow
    ======================
        5,000 Chunks
            ↓
        Retriever
            ↓
        Top 5 Relevant Chunks
            ↓
            LLM
            ↓
          Answer

- Key NOTE:
    - The Retriever's job is finding, while the LLM's job is generating/explaining.
"""

In [ ]:
"""  - Why do we need Retrievers?
    ==============================

let images we have a large knowledge base that contain a pdf of 500 pages and the number of chunk 5000

User asks: What is CNN?

for this user query We don't want to send all 5,000 chunks to the LLM.

That would cause:
    - High token usage
    - Higher cost
    - Slower responses
    - More irrelevant information
    - Potential context-window problems



- Instead of we pass like: 
                    Knowledge Base
                         │
                    5,000 chunks
                         │
                         ▼
                     Retriever
                         │
                  Search & Ranking
                         │
                         ▼
                 5 relevant chunks
                         │
                         ▼
                       LLM
                         │
                         ▼
                      Answer

The primary purpose of a Retriever is:
    - Efficiently retrieve the most relevant information from a large knowledge base.


    - A Retriever is the component that takes a query and retrieves relevant Document objects.
"""

In [ ]:
""" 
         - How Retriever Work Internally
         ==================================

┌─────────────────────────────────────────────────────────────┐
│              RETRIEVER INTERNAL FLOW                        │
│                                                             │
│  1. RECEIVE QUERY                                           │
│     Accept unstructured string input from user or agent     │
│                                                             │
│  2. TRANSFORM & EMBED                                       │
│     Query Translation → Generate variants (MultiQuery)      │
│     Embedding Model   → Convert text to vector [0.12, ...]  │
│                                                             │
│  3. SEARCH & FILTER                                         │
│     Vector Store → Calculate Cosine/Dot-product similarity  │
│     Keyword Index→ Calculate BM25/TF-IDF exact matches      │
│     Metadata     → Apply hard filters (e.g., year > 2020)   │
│     Ensemble     → Merge dense & sparse results via RRF     │
│                                                             │
│  4. POST-PROCESSING (Optional)                              │
│     Re-ranking   → Cross-encoder re-scores top-K results    │
│     Compression  → LLM extracts only relevant sentences     │
│     Parent Lookup→ Map matched chunk ID to full parent doc  │
│                                                             │
│  5. RETURN List[Document]                                   │
│     [Document(page_content="...", metadata={"source":...})] │
└─────────────────────────────────────────────────────────────┘


"""

### 2. Types of Retrievers in LangChain

In [ ]:
""" 
        - The Complete TAXONOMY IN LANGCHAIN
        ========================================

┌──────────────────────────────────────────────────────────────────┐
│                 RETRIEVER TAXONOMY IN LANGCHAIN                  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  VECTOR / DENSE RETRIEVAL (Most Common)                    │  │
│  │  ├── VectorStoreRetriever        → Embedding similarity    │  │
│  │  │   ├── similarity               → Top-k closest chunks   │  │
│  │  │   ├── mmr                      → Relevant + diverse     │  │
│  │  │   └── score_threshold          → Minimum relevance      │  │
│  │  └── TimeWeightedVectorStore      → Similarity + recency   │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  KEYWORD / SPARSE RETRIEVAL                                │  │
│  │  ├── BM25Retriever              → TF-IDF style ranking     │  │
│  │  ├── TFIDFRetriever              → Keyword frequency       │  │
│  │  └── SVMRetriever                → SVM-based relevance     │  │
│  │                                                            │  │
│  │  Best for: Exact names, IDs, error codes, code, acronyms    │ │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  HYBRID / ENSEMBLE RETRIEVAL                               │  │
│  │  ├── EnsembleRetriever          → Combine multiple search  │  │
│  │  │                                 results using RRF       │  │
│  │  ├── BM25 + Vector Retriever    → Keywords + meaning       │  │
│  │  └── External Hybrid Search     → Pinecone, Weaviate,      │  │
│  │                                 Elasticsearch, OpenSearch  │  │
│  │                                                            │  │
│  │  Best for: Enterprise search and production RAG            │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  QUERY-TRANSFORMATION RETRIEVAL                            │  │
│  │  ├── MultiQueryRetriever         → Generates query variants│  │
│  │  ├── RePhraseQueryRetriever      → Rewrites user query     │  │
│  │  ├── SelfQueryRetriever          → Query + metadata filters│  │
│  │  └── HyDE Pattern                → Embeds hypothetical     │  │
│  │                                 answer/document            │  │
│  │                                                            │  │
│  │  Best for: Ambiguous, short, or poorly phrased questions   │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  COMPRESSION / RE-RANKING RETRIEVAL                        │  │
│  │  ├── ContextualCompressionRetriever → Wrapper retriever    │  │
│  │  ├── LLMChainExtractor          → Extract relevant passages│  │
│  │  ├── LLMChainFilter             → Keep/drop whole documents│  │
│  │  ├── EmbeddingsFilter            → Remove weak matches     │  │
│  │  ├── CohereRerank                → Cross-encoder reranking │  │
│  │  ├── FlashrankRerank             → Local/lightweight rerank│  │
│  │  └── BGE Reranker                → Open-source reranking   │  │
│  │                                                            │  │
│  │  Pattern: Retrieve Top-20 → Re-rank/Compress → Return Top-4│  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  HIERARCHICAL / MULTI-REPRESENTATION                       │  │
│  │  ├── ParentDocumentRetriever      → Small chunks → parents │  │
│  │  ├── MultiVectorRetriever         → Many vectors per doc   │  │
│  │  │   ├── Summary vectors          → Search document summary│  │
│  │  │   ├── Hypothetical questions   → Search likely questions│  │
│  │  │   └── Child chunks             → Return original parent │  │
│  │  └── MergerRetriever              → Merge multiple outputs │  │
│  │                                                            │  │
│  │  Best for: Long PDFs, manuals, legal docs, research papers │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  DOCUMENT-SPECIFIC RETRIEVAL                               │  │
│  │  ├── ParentDocumentRetriever      → Preserve parent context│  │
│  │  ├── MultiVectorRetriever         → Summaries/images/tables│  │
│  │  └── Table / SQL Retrievers       → Structured data lookup │  │
│  │                                                            │  │
│  │  Best for: Multi-modal docs, tables, reports, databases    │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  EXTERNAL KNOWLEDGE RETRIEVERS                             │  │
│  │  ├── TavilySearchAPIRetriever     → Live web search        │  │
│  │  ├── WikipediaRetriever           → Wikipedia knowledge    │  │
│  │  ├── ArxivRetriever               → Academic papers        │  │
│  │  ├── PubMedRetriever              → Biomedical literature  │  │
│  │  ├── Google / Bing Search          → Web search providers  │  │
│  │  └── Cloud / SaaS Retrievers       → GitHub, Slack, Notion │  │
│  │                                 Drive, etc.                │  │
│  │                                                            │  │
│  │  Best for: Current, public, or connected third-party data  │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  CUSTOM RETRIEVERS                                         │  │
│  │  ├── BaseRetriever              → Implement custom logic   │  │
│  │  ├── RunnableLambda              → Simple retrieval function│ │
│  │  └── API / Database Retriever    → Connect any data source │  │
│  │                                                            │  │
│  │  Requirement: Input query → Output List[Document]          │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  GOLDEN PRODUCTION STACK:                                        │
│  BM25 + VectorStoreRetriever → EnsembleRetriever → Reranker      │
│  → Contextual Compression → LLM                                  │
└──────────────────────────────────────────────────────────────────┘



"""

### 3. How to create a Retriever in LangChain?

In [ ]:
""" 
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.5,
        "score_threshold": 0.7
    }
)

- Key Note:
    similarity → "Give me the k closest, period."
    mmr → "Give me k results that are relevant AND diverse."
    similarity_score_threshold → "Give me results above a score cutoff."

"""


In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│              HOW TO CREATE A RETRIEVER IN LANGCHAIN         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. CREATE VECTOR STORE                                     │
│                                                             │
│     embeddings = OpenAIEmbeddings()                         │
│                                                             │
│     vector_store = FAISS.from_texts(                        │
│         texts=["doc1", "doc2", "doc3"],                     │
│         embedding=embeddings                                │
│     )                                                       │
│                                                             │
│                         ↓                                   │
│                                                             │
│  2. CREATE RETRIEVER                                        │
│                                                             │
│     retriever = vector_store.as_retriever(                  │
│         search_type="similarity",                           │
│         search_kwargs={"k": 5}                              │
│     )                                                       │
│                                                             │
│                         ↓                                   │
│                                                             │
│  3. SEND QUERY                                              │
│                                                             │
│     docs = retriever.invoke(                                │
│         "What is deep learning?"                            │
│     )                                                       │
│                                                             │
│                         ↓                                   │
│                                                             │
│  4. GET RELEVANT DOCUMENTS                                  │
│                                                             │
│     docs → [Document, Document, Document, ...]              │
│                                                             │
└─────────────────────────────────────────────────────────────┘



"""

###  4. Important Retriever Parameters

In [ ]:
""" 

================================***=====================================


┌─────────────────────────────────────────────────────────────┐
│              IMPORTANT RETRIEVER PARAMETERS                 │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ CORE PARAMETERS (All Retrievers)                      │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ k                  → Number of docs to return         │  │
│  │                      Default: 4                       │  │
│  │                      Range: 1-100+                    │  │
│  │                                                       │  │
│  │ search_type        → Retrieval strategy               │  │
│  │                      "similarity" (default)           │  │
│  │                      "mmr" (diverse results)          │  │
│  │                      "similarity_score_threshold"     │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ SEARCH_KWARGS (Vector Store Retrievers)               │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ fetch_k            → Candidate pool before MMR        │  │
│  │                      Default: 20                      │  │
│  │                      Use: fetch_k >> k (e.g., 20 vs 4)│  │
│  │                                                       │  │
│  │ lambda_mult        → MMR diversity factor             │  │
│  │                      Range: 0.0 - 1.0                 │  │
│  │                      0.0 = Maximum diversity          │  │
│  │                      1.0 = Maximum relevance          │  │
│  │                      Default: 0.5                     │  │
│  │                                                       │  │
│  │ score_threshold    → Minimum similarity score         │  │
│  │                      Range: 0.0 - 1.0                 │  │
│  │                      Use: Filter out weak matches     │  │
│  │                      Example: 0.7 = only strong matches│  │
│  │                                                       │  │
│  │ filter             → Metadata filter (pre-search)     │  │
│  │                      {"year": {"$gte": 2020}}          │  │
│  │                      {"category": "tech"}              │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ ENSEMBLE PARAMETERS                                   │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ retrievers         → List of retriever objects        │  │
│  │                                                       │  │
│  │ weights            → Importance of each retriever     │  │
│  │                      Example: [0.7, 0.3]              │  │
│  │                                                       │  │
│  │ c                  → RRF constant (usually 60)        │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ MULTIQUERY PARAMETERS                                 │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ llm                → LLM to generate query variants   │  │
│  │                                                       │  │
│  │ parser_key         → Key for parsing LLM output       │  │
│  │                      Default: "lines"                  │ │
│  │                      Options: "lines", "json", "pydantic"│  
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ CONTEXTUAL COMPRESSION PARAMETERS                     │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ base_retriever     → Underlying retriever to wrap     │  │
│  │                                                       │  │
│  │ base_compressor    → Compression strategy             │  │
│  │                      LLMChainExtractor                │  │
│  │                      LLMChainFilter                   │  │
│  │                      EmbeddingsFilter                 │  │
│  │                      CohereRerank(top_n=4)            │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ PARENT DOCUMENT PARAMETERS                            │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ parent_splitter    → How to split parent docs         │  │
│  │                      chunk_size: 1000-2000            │  │
│  │                                                       │  │
│  │ child_splitter     → How to split for embedding       │  │
│  │                      chunk_size: 100-400              │  │
│  │                                                       │  │
│  │ docstore           → Storage for parent docs          │  │
│  │                      InMemoryStore() (dev)            │  │
│  │                      RedisStore() (prod)              │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ TIME-WEIGHTED PARAMETERS                              │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ decay_rate         → How fast old docs lose weight    │  │
│  │                      Range: 0.0 - 1.0                 │  │
│  │                      0.01 = slow decay                │  │
│  │                      0.1 = fast decay                 │  │
│  │                                                       │  │
│  │ hours_passed       → Internal time delta (auto)       │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  PRO TIP:                                                   │  
│  Always test k and score_threshold together —               │
│  if k=10 but only 2 results score above 0.7,                │
│  you're feeding noise to your LLM!                          │
└─────────────────────────────────────────────────────────────┘


"""

### 5. Retriever Use Cases

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│                 RETRIEVER USE CASES                         │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 1. QUESTION ANSWERING (RAG)                           │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ Scenario: Chat with your PDFs / Docs / Knowledge Base │  │
│  │                                                       │  │
│  │ User: "What is our refund policy?"                    │  │
│  │   ↓                                                   │  │
│  │ Retriever → fetches relevant policy sections          │  │
│  │   ↓                                                   │  │
│  │ LLM → answers using ONLY retrieved docs               │  │
│  │                                                       │  │
│  │ Best Retriever: VectorStoreRetriever (k=4)           │  │
│  │ Industries: Legal, HR, Finance, Education             │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 2. SEMANTIC SEARCH ENGINE                             │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ Scenario: Search by MEANING not keywords              │  │
│  │                                                       │  │
│  │ Query: "How do I fix a slow computer?"                │  │
│  │ Finds: "Performance optimization techniques..."       │  │
│  │ (No keyword overlap — pure semantic match!)           │  │
│  │                                                       │  │
│  │ Best Retriever: Ensemble (BM25 + Vector)              │  │
│  │ Industries: E-commerce, Enterprise Search, Media      │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 3. CHATBOT MEMORY & HISTORY                           │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ Scenario: Long conversations with relevant recall     │  │
│  │                                                       │  │
│  │ User mentions preference in message #5               │  │
│  │ At message #50, bot recalls it via retrieval          │  │
│  │                                                       │  │
│  │ Best Retriever: TimeWeightedVectorStoreRetriever      │  │
│  │ decay_rate=0.01 → recent context matters more         │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 4. CUSTOMER SUPPORT AUTOMATION                        │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ Scenario: Answer tickets using past resolutions       │  │
│  │                                                       │  │
│  │ Ticket: "Login button not working on Safari"          │  │
│  │   ↓                                                   │  │
│  │ Retriever → finds 3 similar past tickets + fixes     │  │
│  │   ↓                                                   │  │
│  │ Agent/Copilot → suggests proven solution              │  │
│  │                                                       │  │
│  │ Best Retriever: Hybrid + Re-ranking                   │  │
│  │ (Error codes need keywords; descriptions need semantic)│  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 5. CODE SEARCH & ASSISTANCE                           │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ Scenario: "Find where we validate user input"         │  │
│  │                                                       │  │
│  │ Needs: Function names (BM25) + purpose (semantic)     │  │
│  │                                                       │  │
│  │ Best Retriever: EnsembleRetriever                     │  │
│  │ BM25 → catches validate_input(), check_form()         │  │
│  │ Vector → catches "ensures data integrity" comments    │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 6. RECOMMENDATION SYSTEMS                             │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ Scenario: "More like this product/article"            │  │
│  │                                                       │  │
│  │ Item description → Embedding → Similar items          │  │
│  │                                                       │  │
│  │ Best Retriever: VectorStoreRetriever (mmr)            │  │
│  │ MMR ensures DIVERSE recommendations, not 5 variants   │  │
│  │ of the same thing                                     │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 7. STRUCTURED DATA QUERIES                            │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ Scenario: "Show me red sneakers under $100"           │  │
│  │                                                       │  │
│  │ SelfQueryRetriever auto-decomposes:                   │  │
│  │   Semantic: "sneakers"                                │  │
│  │   Filter: color="red" AND price < 100                 │  │
│  │                                                       │  │
│  │ Best Retriever: SelfQueryRetriever                    │  │
│  │ Industries: E-commerce, Real Estate, Job Portals      │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 8. RESEARCH & LITERATURE REVIEW                       │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ Scenario: "Recent papers on transformer efficiency"   │  │
│  │                                                       │  │
│  │ Best Retriever: ArxivRetriever + ParentDocument       │  │
│  │                                                       │  │
│  │ ParentDocument: match on abstract (small),            │  │
│  │                 return full paper section (big)       │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 9. AGENT TOOL USE (Function-Calling)                  │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ Scenario: Agent decides WHEN and WHAT to retrieve     │  │
│  │                                                       │  │
│  │ Agent: "I need more info on X" → retriever.invoke(X)  │  │
│  │                                                       │  │
│  │ Pattern: Retriever as a Tool in ReAct/Agent loop      │  │
│  │ Advanced: Self-RAG, CRAG (corrective retrieval)       │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 10. COMPLIANCE & AUDIT TRAIL                          │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ Scenario: Every answer must cite sources              │  │
│  │                                                       │  │
│  │ Retrieved docs carry metadata:                        │  │
│  │   {"source": "policy_v3.pdf", "page": 42}             │  │
│  │                                                       │  │
│  │ LLM answers WITH citations → full traceability        │  │
│  └───────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────┘


"""

### 6. Advantages and Limitations of Retrievers

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           ADVANTAGES OF RETRIEVERS                          │
│                                                             │
│  ✅ 1. OVERCOMES CONTEXT WINDOW LIMITS                      │
│     Don't stuff 10,000 pages into prompt —                  │
│     fetch only the 4 relevant chunks                        │
│                                                             │
│  ✅ 2. REDUCES HALLUCINATION                                │
│     LLM grounds answers in real retrieved documents         │
│     → factual, verifiable, citable responses                │
│                                                             │
│  ✅ 3. COST EFFICIENCY                                      │
│     Fewer tokens = cheaper API calls                        │
│     Retrieval is cheap; generation is expensive             │
│                                                             │
│  ✅ 4. ALWAYS-CURRENT DATA                                  │
│     No retraining needed — just add new docs to index       │
│     Update knowledge in seconds, not weeks                  │
│                                                             │
│  ✅ 5. PRIVATE DATA ACCESS                                  │
│     Query internal docs, DBs, wikis securely                │
│     Data never trains the LLM                               │
│                                                             │
│  ✅ 6. COMPOSABLE & MODULAR                                 │
│     All retrievers share invoke() interface                 │
│     Swap/stack like Lego: BM25 + Vector + Reranker          │
│                                                             │
│  ✅ 7. TRANSPARENCY & DEBUGGING                             │
│     You can SEE what was retrieved before generation        │
│     Separates retrieval errors from LLM errors              │
│                                                             │
│  ✅ 8. LOW LATENCY AT SCALE                                 │
│     Vector search: milliseconds over millions of docs       │
│     No inference needed (unlike LLM generation)             │
│                                                             │
│  ✅ 9. ASYNC + STREAMING NATIVE                             │
│     Runnable interface → ainvoke(), batch(), stream()       │
│     Fits modern LCEL pipelines out of the box               │
│                                                             │
│  ✅ 10. SOURCE ATTRIBUTION                                  │
│      Metadata flows through → every answer is traceable     │
│      Critical for legal/medical/enterprise use              │
└─────────────────────────────────────────────────────────────┘

"""

In [ ]:
"""

┌─────────────────────────────────────────────────────────────┐
│           LIMITATIONS OF RETRIEVERS                         │
│                                                             │
│  ❌ 1. RETRIEVAL QUALITY IS THE CEILING                     │
│     If the right doc isn't retrieved,                       │
│     the LLM CANNOT answer correctly                         │
│     "Garbage in → garbage out" applies here                 │
│                                                             │
│  ❌ 2. CHUNKING SENSITIVITY                                 │
│     Bad chunking = broken context                           │
│     Answer split across two chunks = partial answers        │
│     The "lost in the middle" problem for large chunks       │
│                                                             │
│  ❌ 3. EMBEDDING MODEL DEPENDENCE                           │
│     Retrieval only as good as the embedding model           │
│     Domain mismatch (medical text + general embedder)       │
│     Query and docs MUST use the same embedding model        │
│                                                             │
│  ❌ 4. NO REASONING ACROSS DOCUMENTS                        │
│     Retrieval finds SIMILAR content                         │
│     Multi-hop questions ("Compare A and B from             │
│     different sources") need multiple retrievals           │
│                                                            │
│  ❌ 5. LATENCY OVERHEAD (Complex Stacks)                   │
│     MultiQuery → 1 query becomes 5 LLM calls               │
│     Compression → LLM call per document                    │
│     Reranking → cross-encoder scoring per doc              │
│     Simple (fast) vs Advanced (accurate) tradeoff          │
│                                                             │
│  ❌ 6. METADATA MAINTENANCE BURDEN                          │
│     SelfQueryRetriever needs accurate metadata schemas      │
│     Filters need clean, consistent metadata upfront         │
│                                                             │
│  ❌ 7. SCALAR SEMANTIC SIMILARITY LIMITS                    │
│     Cosine similarity ≠ true relevance                      │
│     Nuance, negation, and temporal logic are hard          │
│     ("What is NOT covered?" → retrievers struggle)        │
│                                                            │
│  ❌ 8. INDEX FRESHNESS / SYNC ISSUES                       │
│     Stale index = stale answers                            │
│     Need re-indexing pipelines for updates                │
│                                                             │
│  ❌ 9. EVALUATION IS HARD                                   │
│     "Relevant" is subjective                               │
│     Needs golden datasets + metrics (MRR, Hit Rate)       │
│     Retrieval errors are often silent                      │
│                                                             │
│  ❌ 10. COST OF ADVANCED FEATURES                           │
│      MultiQuery/Compression = extra LLM calls per query   │
│      Rerankers = additional API/service costs             │
│      Production hybrid stacks get expensive at scale      │
└─────────────────────────────────────────────────────────────┘


┌─────────────────────────────────────────────────────────────┐
│           MITIGATION STRATEGIES (QUICK REFERENCE)          │
│                                                             │
│  LIMITATION              →  SOLUTION                        │
│  ─────────────────────────────────────────────────────      │
│  Retrieval ceiling       →  Hybrid search + re-ranking      │
│  Chunking issues         →  ParentDocumentRetriever         │
│  Embedding mismatch      →  Domain-specific embedders       │
│  Multi-hop questions     →  Agentic/iterative retrieval     │
│  Latency of MultiQuery   →  Cache generated query variants  │
│  Stale index             →  Incremental upsert pipelines    │
│  Hard evaluation         →  RAGAS + LangSmith tracing       │
│                                                             │
│  ⚡ BOTTOM LINE:                                            │
│  Retrieval quality CAPS your RAG system quality.           │
│  Invest in retrieval FIRST, LLM prompt engineering SECOND. │
└─────────────────────────────────────────────────────────────┘
```

"""

### 7. Summary of Retrievers

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                         RETRIEVERS                               │
│                                                                  │
│  WHAT:  Fetch relevant Documents given a user query              │
│  WHY:   Plain vector search fails on phrasing, keywords, & noise │
│  WHERE: After Vector Store, before Prompt/LLM in the RAG chain   │
│                                                                  │
│  KEY PARAMS:                                                     │
│    search_type   → "similarity", "mmr", "score_threshold"        │
│    k             → Number of final documents to return           │
│    fetch_k       → Docs fetched before MMR diversity filtering   │
│                                                                  │
│  TOP RETRIEVERS:                                                 │
│    -  vectorstore.as_retriever()      → Basic semantic search    │
│    -  MultiQueryRetriever             → LLM expands query        │
│    -  ContextualCompressionRetriever  → Removes chunk junk       │
│    -  EnsembleRetriever               → Hybrid (BM25 + Vector)   │
│    -  ParentDocumentRetriever         → Small search, big context│
│    -  SelfQueryingRetriever           → Auto-extracts metadata   │
│                                                                  │
│  DEFAULT RECOMMENDATION:                                         │
│    retriever = vectorstore.as_retriever(                         │
│        search_type="mmr",                                        │
│        search_kwargs={"k": 4, "fetch_k": 20}                     │
│    )                                                             │
│                                                                  │
│  PIPELINE:                                                       │
│    [Load] → [Split] → [Embed] → [VecStore] → [Retriever] → [LLM] │
│                                                ↑                 │
│                                          THIS COMPONENT          │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Never ship basic similarity in prod. Combine Ensemble (Hybrid) │
│   with ContextualCompression for robust, noise-free retrieval."  │
└──────────────────────────────────────────────────────────────────┘


"""